Author: Yashasvi Moon, email: ymoon@illinois.edu

Adapted for cosmicAI by Padma Venkatraman

# Introduction
With the rise of large photometric surveys like the Rubin Observatory, spectroscopically followed up transients will be limited to < 1%. It will be essential to classify transients preliminarily but confidently before they fully evolve to trigger spectroscopic follow-up for exotic transients.

Tidal disruption events (TDEs) are relatively rarer transients that occur when a star gets too close to a blackhole (BH) and is pulled apart by the BH's tidal force resulting in a temporary accretion disk around the BH. TDEs help us probe BH masses and can be used to find lower mass BHs.

TDEs are **overrepresented** in galaxies that have had a quench in star formation such as quiescent Balmer-strong galaxies (QBS) according to French+2016. Cutoffs using certain spectral lines (eg H-alpha and lick H-delta) are used to detect these unusual galaxies. But with Rubin, we will only get u,g,r,i,z,y photometric data. The goal of this project is to translate these spectroscopic cutoffs to photometry so that we can leverage host galaxy information to classify TDEs using machine learning methods.

This project is published here: https://iopscience.iop.org/article/10.3847/2515-5172/ad9822 (Moon and French et al 2024). Feel free to look through the paper to understand more!


# Preparing the QBS Galaxy Dataset

Step 1: Prepare the photometric dataset used for machine learning classification of TDE host galaxies aka quiescent Balmer-strong (QBS) galaxies.

The pre-queried dataset contains follwoing host galaxy information:
- optical and infrared photometry
-  spectroscopic information
-  seric index
-  photometric redshift

In this notebook, we will:
- Start with a pre-queried galaxy catalog
- filter non-physical values
- create photometric color features that will be used for training along with photo-z and sersic index
- create QBS labels (0 stands for not QBS and 1 stands for QBS) to perform supervised learning

## Import Libraries

In [7]:
import os
os.getcwd()
os.listdir(os.getcwd())

['.config', 'drive', 'sample_data']

In [9]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
import os
from google.colab import drive
drive.mount('/content/drive')
### change this line
notebook_data_location = '/content/drive/'
os.chdir(notebook_data_location)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Remove non-physical data
Rows containing NaN or infinite values are removed before training.

In [11]:
os.listdir(os.getcwd())

['Shareddrives', '.shortcut-targets-by-id', 'MyDrive', '.Trash-0']

In [10]:
datadir = 'data'
pardata = pd.read_csv(os.path.join(datadir,'df_matched_updated.csv'))

FileNotFoundError: [Errno 2] No such file or directory: 'data/df_matched_updated.csv'

In [ ]:
pardata_cleaned = ...

Look at the columns! What does each column describe? Discuss with neighbors, ask your TA etc.

## Apply Redshift Selection

Limit the sample to galaxies with **photometric redshift below 0.3**:
Why did we apply this cut? (Hint: look at the paper!) You could experiment with other cuts too!

In [ ]:
pardata_filtered = ...


In [ ]:
### look at your filtered data
display(pardata_filtered)

,ls_id,ra,dec,mag_g,mag_i,mag_r,mag_z,mag_w1,mag_w2,mag_w3,sersic,z_phot_median,lick_hd_a,lick_hd_a_err,h_alpha_eqw,h_alpha_flux
0,10995417315544404,316.194161,-7.745349,17.012596,15.913244,16.275385,15.677995,15.750366,16.046907,14.249937,2.614517,0.115445,2.838780,0.897309,-11.287642,177.963074
1,10995417315549631,316.288753,-7.744508,17.796990,16.554451,16.933170,16.302767,16.453445,17.019648,15.559782,6.000000,0.070549,3.199646,0.750215,-5.666849,107.611938
2,10995417316593073,316.442750,-7.759569,16.786205,15.456808,15.861318,15.192717,15.322462,15.812057,14.978994,1.951873,0.089307,-0.151788,0.869566,-5.631099,162.482422
3,10995415820273584,316.671305,-7.916997,17.170082,15.934116,16.367483,15.700871,15.534376,15.634947,14.155700,6.000000,0.120236,0.829948,0.626013,-25.850813,528.533508
4,10995415820274346,316.685902,-7.925625,18.247866,17.048077,17.440180,16.824520,16.731937,16.981560,15.202742,2.143291,0.145451,4.038033,0.827559,-20.598230,258.365356
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
179122,10995467032727578,314.637082,0.403892,18.213003,16.837880,17.259504,16.558014,16.705145,17.151380,17.284990,2.643269,0.103209,-0.942777,1.323242,-4.371205,61.526470
179123,10995465522780389,314.711123,0.342184,19.213675,17.669874,18.131296,17.354496,17.022440,16.741398,15.632656,2.928447,0.207379,1.448538,1.196925,-12.777702,114.671516
179124,10995465523823262,314.768361,0.339075,18.473130,17.498838,17.840406,17.339205,17.403399,17.667376,15.738592,0.918556,0.148061,4.091952,0.943455,-29.674618,187.204086
179125,10995465523823262,314.768361,0.339075,18.473130,17.498838,17.840406,17.339205,17.403399,17.667376,15.738592,0.918556,0.148061,4.091952,0.943455,-29.674618,187.204086


## Create photometric color features
We do not use photometric bands as features, rather we use the difference between consecutive photometric bands, also known as color.

In [ ]:
# create columns for all color: colors are the difference of two neighboring magnitudes
# g-r color
...


## Add conditions for QBS galaxies
According to pre-determined criterion for QBS galaxies (French&Zabludoff 2018):
- QBS galaxies have a low H-alpha signal which stands for low to none star formation
- QBS galaxies have ligh h-delta llick index which indicates that type A stars have recently supernovaed.

- Together, these clues point toward recent quench in star formation

![My Figure](QBS_french.png)

In [ ]:
#add the conditions described above but more quantitatively again using (French&Zabludoff 2018):
conditions = (pardata_filtered['lick_hd_a'] > 1.3) & (pardata_filtered['h_alpha_eqw'] > -5) & (pardata_filtered['h_alpha_flux'] > 0) & (pardata_filtered['h_alpha_flux'] < 1000)
pardata_filtered['labels'] = np.where(conditions, 1, 0)


In [ ]:
num_labeled_1 = (pardata_filtered['labels'] == 1).sum()
print(f"Number of QBS galaxies in data {num_labeled_1}")

Number of QBS galaxies in data 5948


In [ ]:
#export and save the dataset
pardata_filtered.to_csv('labelled_dataset.csv', index=False)

### Here, you can do your own exploratory data analysis!

Look at the different columns. Use `seaborn` to make a giant corner plot. Which properties are correlated? Which columns have many outliers? Where do outliers completely change the shape of the distribution?